# Notebook 04 — HITRAN-Based CO₂ Spectrum

## Overview
This notebook uses a HITRAN-style line list to generate a realistic CO₂ absorption spectrum.

## HITRAN Database
The [HITRAN database](https://hitran.org/) provides spectroscopic parameters for 55+ molecules. For each line transition it provides:

| Parameter | Symbol | Units |
|-----------|--------|-------|
| Line centre | $\nu_0$ | cm⁻¹ |
| Line strength at 296 K | $S(T_{\text{ref}})$ | cm/molecule |
| Air-broadened HWHM | $\gamma_{\text{air}}$ | cm⁻¹/atm |
| Lower-state energy | $E''$ | cm⁻¹ |
| Temperature exponent | $n_{\text{air}}$ | — |

## Line Strength Temperature Correction
$$S(T) = S(T_{\text{ref}}) \cdot \exp\left(-\frac{hcE''}{k_B}\left(\frac{1}{T} - \frac{1}{T_{\text{ref}}}\right)\right) \cdot \frac{1 - e^{-hc\nu_0/k_BT}}{1 - e^{-hc\nu_0/k_BT_{\text{ref}}}}$$

In [ ]:
import sys, os

# ── Robust path fix ────────────────────────────────────────────────────────
# Works whether Jupyter is launched from the project root, the notebooks/
# folder, or anywhere else.  Walks up from __file__ (or cwd as fallback)
# until it finds the src/ directory that contains hitran_model.py.
def _add_src_to_path():
    try:
        _start = os.path.dirname(os.path.abspath(__file__))
    except NameError:                      # __file__ undefined in Jupyter
        _start = os.getcwd()
    _candidate = _start
    for _ in range(6):
        _src = os.path.join(_candidate, "src")
        if os.path.isdir(_src) and os.path.isfile(os.path.join(_src, "hitran_model.py")):
            if _src not in sys.path:
                sys.path.insert(0, _src)
            return _src
        _candidate = os.path.dirname(_candidate)
    raise RuntimeError(
        f"Cannot find src/ directory.  Run Jupyter from the project root:\n"
        f"  cd co2_retrieval_simulator && jupyter notebook"
    )

_SRC = _add_src_to_path()
print(f"src/ found at: {_SRC}")
# ───────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
from hitran_model import (
    SYNTHETIC_CO2_LINES,
    hitran_cross_section,
    correct_line_strength,
    plot_hitran_spectrum,
    plot_spectrum_sensitivity,
    HAPI_AVAILABLE,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print(f'HAPI available: {HAPI_AVAILABLE}')
print(f'Synthetic line list: {len(SYNTHETIC_CO2_LINES)} lines')

## 1. Inspect the Line List

In [ ]:
import pandas as pd

df = pd.DataFrame(SYNTHETIC_CO2_LINES,
                  columns=['nu0 [cm⁻¹]', 'S(296K) [cm/mol]', 'γ_air [cm⁻¹/atm]', "E'' [cm⁻¹]", 'n_air'])
df = df.sort_values('nu0 [cm⁻¹]').reset_index(drop=True)
print(df.to_string(index=False, float_format='{:.3e}'.format))

## 2. Line Strength Temperature Correction

In [ ]:
# Show how line strength varies with temperature for several lines
temps = np.linspace(180, 320, 200)

fig, ax = plt.subplots(figsize=(9, 4))

for i in [3, 7, 15]:  # pick a few lines
    nu0_i = SYNTHETIC_CO2_LINES[i, 0]
    S_ref = SYNTHETIC_CO2_LINES[i, 1]
    E_pp  = SYNTHETIC_CO2_LINES[i, 3]
    S_T   = [correct_line_strength(np.array([S_ref]), np.array([nu0_i]),
                                   np.array([E_pp]), T)[0] for T in temps]
    ax.plot(temps, np.array(S_T) / S_ref, lw=2,
            label=f'ν₀={nu0_i:.1f} cm⁻¹, E\"={E_pp:.0f} cm⁻¹')

ax.axvline(296, color='gray', ls='--', lw=1, label='T_ref = 296 K')
ax.set_xlabel('Temperature [K]'); ax.set_ylabel('S(T) / S(296 K)')
ax.set_title('Line Strength Temperature Correction')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/04a_line_strength_temperature.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Full HITRAN Absorption Spectrum

Surface conditions: T = 288 K, P = 1 atm.

In [ ]:
nu = np.linspace(6210, 6275, 6000)
P  = 101325.0  # Pa
T  = 288.0     # K

xsec = hitran_cross_section(nu, SYNTHETIC_CO2_LINES, P, T)

print(f'Peak cross section: {xsec.max():.3e} cm²/molecule')
print(f'Spectral range:     {nu.min():.0f} – {nu.max():.0f} cm⁻¹')

plot_hitran_spectrum(nu, xsec, SYNTHETIC_CO2_LINES,
                    savefig='../figures/04b_hitran_spectrum.png')

## 4. Pressure-Level Spectra

Cross section at different atmospheric pressure levels to illustrate pressure broadening.

In [ ]:
levels = {
    'Surface (1013 hPa)': (101325, 288),
    '500 hPa':            (50000,  255),
    '200 hPa':            (20000,  225),
    '50 hPa':             (5000,   210),
}

nu_zoom = np.linspace(6228, 6242, 3000)
fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(levels)))

for (label, (P_i, T_i)), col in zip(levels.items(), colors):
    xs = hitran_cross_section(nu_zoom, SYNTHETIC_CO2_LINES, P_i, T_i)
    ax.plot(nu_zoom, xs, color=col, lw=1.5, label=label)

ax.set_xlabel('Wavenumber [cm⁻¹]'); ax.set_ylabel('Cross section [cm²/molecule]')
ax.set_title('Pressure Broadening: CO₂ Cross Section at Different Levels')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/04c_pressure_broadening.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sensitivity to CO₂ Concentration

In [ ]:
plot_spectrum_sensitivity(
    nu, SYNTHETIC_CO2_LINES,
    co2_concentrations_ppm=[350, 380, 400, 420, 450, 500],
    pressure_Pa=101325.0, temperature_K=288.0,
    savefig='../figures/04d_co2_sensitivity.png'
)

## 6. HAPI Integration (Optional)

If the `hitran-api` package is installed and internet access is available,
you can download the official HITRAN data for CO₂:

```python
from hitran_model import download_hitran_co2, hapi_cross_section

# Download CO₂ lines for the 1.6 µm band
download_hitran_co2(6200, 6280, data_dir='../data')

# Compute cross section using HAPI
xsec_hapi = hapi_cross_section(nu, pressure_atm=1.0, temperature_K=288.0)
```

The HAPI-based spectrum includes thousands of lines and will closely match
the spectra used in real satellite retrieval algorithms.